In [ ]:
import os
from pathlib import Path

# Definir la ruta exacta de next_7
NEXT7_DIR = Path("/Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/ninth_container/repaso/next_7")
NEXT7_DIR.mkdir(parents=True, exist_ok=True)

# 1. Crear app.py
app_code = '''import random
from datetime import datetime, timedelta
from pathlib import Path
import sys

BASE_DIR = Path.cwd()
INPUT_DIR = BASE_DIR / "data" / "input"
OUTPUT_DIR = BASE_DIR / "data" / "output"

try:
    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"📁 Entorno listo en: {BASE_DIR}")
except Exception as e:
    print(f"❌ Error al crear carpetas: {e}")
    sys.exit(1)

input_file_path = INPUT_DIR / "raw_fleet_telemetry.csv"
output_file_path = OUTPUT_DIR / "clean_critical_incidents.csv"

start_time = datetime.now()
fault_codes = ["P0171", "P0300", "P0420", "NONE", "NONE", "NONE", "NONE"]

print("⚡ Generando 10,000 registros de telemetría...")

with open(input_file_path, "w", encoding="utf-8") as f:
    f.write("timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\\n")
    for i in range(10000):
        timestamp = start_time + timedelta(seconds=i)
        v_id = f"VIN-{random.randint(0, 999999):06d}"
        rpm = random.randint(1500, 3500) if random.random() > 0.05 else 0  
        temp = random.randint(85, 105)
        code = random.choice(fault_codes)
        f.write(f"{timestamp},{v_id},{rpm},{temp},{code}\\n")

print(f"📝 Archivo listo: '{input_file_path.name}'")

def telemetry_streamer(file_path: Path):
    with open(file_path, "r", encoding="utf-8") as f:
        f.readline()
        for line in f:
            if not line.strip():
                continue
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

def telemetry_writer(output_path: Path, processed_entry):
    with open(output_path, "a", encoding="utf-8") as f:
        line = (
            f"{processed_entry['timestamp']},"
            f"{processed_entry['vehicle_id']},"
            f"{processed_entry['engine_rpm']},"
            f"{processed_entry['coolant_temp_c']},"
            f"{processed_entry['fault_code']},"
            f"{processed_entry['alert_status']},"
            f"{processed_entry['processed_at']}\\n"
        )
        f.write(line)

if input_file_path.exists():
    with open(output_file_path, "w", encoding="utf-8") as f:
        f.write("timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\\n")
    
    print("\\n🚀 Procesando lote de datos...")
    stream = telemetry_streamer(input_file_path)
    count = 0
    for raw in stream:
        enriched = telemetry_processor(raw)
        if enriched:
            telemetry_writer(output_file_path, enriched)
            count += 1
    print(f"🏁 Proceso completado: {count} incidentes guardados.")
'''

with open(NEXT7_DIR / "app.py", "w") as f:
    f.write(app_code)

# 2. Crear test_app.py
test_code = '''import pytest
from app import telemetry_processor

def test_telemetry_processor_ignores_none():
    sample_none = {
        "timestamp": "2026-07-30 12:00:00",
        "vehicle_id": "VIN-123456",
        "engine_rpm": 2200,
        "coolant_temp_c": 90,
        "fault_code": "NONE"
    }
    assert telemetry_processor(sample_none) is None

def test_telemetry_processor_detects_fault():
    sample_fault = {
        "timestamp": "2026-07-30 12:00:00",
        "vehicle_id": "VIN-654321",
        "engine_rpm": 2800,
        "coolant_temp_c": 98,
        "fault_code": "P0300"
    }
    result = telemetry_processor(sample_fault)
    assert result is not None
    assert result["fault_code"] == "P0300"
    assert "alert_status" in result
'''

with open(NEXT7_DIR / "test_app.py", "w") as f:
    f.write(test_code)

# 3. Crear requirements.txt
with open(NEXT7_DIR / "requirements.txt", "w") as f:
    f.write("pytest\n")

# 4. Crear Dockerfile
docker_code = '''FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

CMD ["python", "app.py"]
'''

with open(NEXT7_DIR / "Dockerfile", "w") as f:
    f.write(docker_code)

print("✅ Todos los archivos para next_7 han sido creados correctamente.")

In [1]:
import os
from pathlib import Path

# Definir la ruta exacta de next_7
NEXT7_DIR = Path("/Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/ninth_container/repaso/next_7")
NEXT7_DIR.mkdir(parents=True, exist_ok=True)
